In [18]:
import os, sys

# Make sure we're in the project root
os.chdir("/Users/danesh/Documents/GitHub/trueQ")

# Ensure project root is on sys.path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from config import *
from noise_models import *
from cer import *

In [74]:
noisy_sim = sim_gate_ind[2][0]
noisy_gates = noisy_gates_dict(noisy_sim)

In [34]:
def get_dressed_cycles(circuit: tq.Circuit):

    return [circuit[k:k+2] for k in range(circuit.n_cycles//2 -1)] + [[circuit[-2]]]

In [78]:
circ = tq.Circuit(3*[{0:Gate.x, 1:Gate.x, 2:Gate.s},{0:Gate.h, (1,2) : Gate.cx}]+[{0:Gate.i, 1:Gate.i, 2:Gate.y}])
circ.measure_all()

Circuit(Cycle((0,): Gate.x, (1,): Gate.x, (2,): Gate.s),Cycle((0,): Gate.h, (1, 2): Gate.cx),Cycle((0,): Gate.x, (1,): Gate.x, (2,): Gate.s),Cycle((0,): Gate.h, (1, 2): Gate.cx),Cycle((0,): Gate.x, (1,): Gate.x, (2,): Gate.s),Cycle((0,): Gate.h, (1, 2): Gate.cx),Cycle((0,): Gate.id, (1,): Gate.id, (2,): Gate.y),Cycle((0,): Meas(), (1,): Meas(), (2,): Meas(), marker=1))

In [53]:
circ.draw()

DisplayWrapper(<svg xmlns="http://w...)

In [79]:
dressed_cycle = tq.Circuit(get_dressed_cycles(circ)[0])

In [85]:
hard_cycle = dressed_cycle[-1]
easy_cycle = dressed_cycle[0]

In [92]:
lst = []
for qubit, hard_gate in hard_cycle.gates.items():
    
    if hard_gate in [Gate.h, Gate.t]:
        easy_gate = easy_cycle.gates[qubit]
        lst.append(effective_dressed_gate(hard_gate, easy_gate, noisy_gates=noisy_gates))
        print(f"Qubit {qubit}: {hard_gate} over {easy_gate}")

    if hard_gate == Gate.cx:
        easy_gate_0 = easy_cycle.gates[(qubit[0],)]
        easy_gate_1 = easy_cycle.gates[(qubit[1],)]
        lst.append(effective_dressed_gate(hard_gate, tq.Gate(np.kron(easy_gate_0.mat, easy_gate_1.mat)), noisy_gates=noisy_gates))
        print(f"Qubits {qubit}: {hard_gate} over ({easy_gate_0}, {easy_gate_1})")

Qubit (0,): Gate.h over Gate.x
Qubits (1, 2): Gate.cx over (Gate.x, Gate.s)


In [94]:
len(lst)

2